<img src="https://raw.githubusercontent.com/ComplianceAnalytics/aml-book1/main/assets/cal_logo_banner.png" alt="Compliance Analytics Ltd" width="300" onerror="this.style.display='none'">

# Applied AML Analytics: Turning Data Science Skills into Compliance Decisions
## Chapter 6 — Segmentation
### Companion Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_06.ipynb)

---

**Book:** *Applied AML Analytics: Turning Data Science Skills into Compliance Decisions* — Book 1  
**Publisher companion repository:** [github.com/ComplianceAnalytics/aml-book1](https://github.com/ComplianceAnalytics/aml-book1)  
**Dataset:** Northgate Retail Bank (synthetic — all data is fictional)  
**Chapters covered:** 3 · 4 · 5 · 6 · 7 · **5 (this notebook)**

> **How to use this notebook**  
> Run cells top-to-bottom using **Shift+Enter** or the ▶ button. The setup cell (Section 0) must run first — it generates the Northgate dataset that all later cells depend on. You do not need to install anything; all required libraries are pre-installed in Google Colab.

---

## Contents

| Section | Description | Exercise link |
|---------|-------------|---------------|
| **0. Setup** | Generate the Northgate dataset | — |
| **1. Colab Preview** | K-Means clustering on behavioural features · segment labelling (mirrors Section 6.9 of the text) | — |
| **2. Exercise 6.1 Extension** | Elbow method · segment characterisation · mule cluster analysis | Exercise 6.1 |
| **3. Reflection cells** | Structured answer prompts | Exercise 6.1 |

---
## Section 0 — Setup: Generate the Northgate Dataset

**Run this cell first.** It generates four CSV files in the Colab session's working directory:

| File | Rows | Description |
|------|------|-------------|
| `nb_transactions.csv` | ~23,000 | All account transactions, Jan–Dec 2023 |
| `nb_customers.csv` | 500 | Customer and account records |
| `nb_counterparties.csv` | 300 | Counterparty firms and their country codes |
| `nb_accounts.csv` | 500 | Account metadata |

The dataset is **fully synthetic**. Northgate Retail Bank does not exist. All account IDs, names, and transactions are generated from a fixed random seed for educational purposes only.

In [ ]:
import numpy as np
import pandas as pd
from datetime import date, timedelta

rng = np.random.default_rng(42)

# ── Counterparties ────────────────────────────────────────────────────────────
HIGH_RISK  = ['KP', 'IR', 'MM', 'SY', 'YE', 'AF', 'LY']
LOW_RISK   = ['US', 'GB', 'DE', 'FR', 'CA', 'AU', 'SG', 'JP', 'NL', 'CH']

n_cpty      = 300
cpty_ids    = [f'CPT{i:04d}' for i in range(1, n_cpty + 1)]
cpty_cc     = (rng.choice(HIGH_RISK, size=30).tolist() +
               rng.choice(LOW_RISK,  size=270, replace=True).tolist())
rng.shuffle(cpty_cc)
df_cpty = pd.DataFrame({'counterparty_id': cpty_ids, 'country_code': cpty_cc})
df_cpty.to_csv('nb_counterparties.csv', index=False)

n_cust   = 500
cust_ids = [f'NRB_{i:03d}' for i in range(1, n_cust + 1)]
acct_ids = [f'ACC{i:04d}' for i in range(1, n_cust + 1)]
mule_idx = list(range(6))

occupations = ['Employed', 'Self-Employed', 'Retired', 'Student', None]
occ_probs   = [0.55, 0.20, 0.12, 0.08, 0.05]
crr_scores  = rng.choice([1,2,3,4,5], p=[0.35,0.30,0.20,0.10,0.05], size=n_cust)
for i in mule_idx:
    crr_scores[i] = rng.choice([3,4])
incomes_k = rng.lognormal(mean=3.1, sigma=0.5, size=n_cust) * 1000
for i in mule_idx:
    incomes_k[i] = rng.uniform(18, 24) * 1000

df_cust = pd.DataFrame({
    'customer_id':       cust_ids,
    'account_id':        acct_ids,
    'occupation':        rng.choice(occupations, p=occ_probs, size=n_cust),
    'crr_score':         crr_scores,
    'stated_income_usd': np.round(incomes_k, -2),
    'account_open_date': [
        (date(2020,1,1) + timedelta(days=int(d))).isoformat()
        for d in rng.integers(0, 1460, size=n_cust)
    ],
})
df_cust.to_csv('nb_customers.csv', index=False)
df_cust.to_csv('nb_accounts.csv',  index=False)

txn_rows = []
start    = date(2023, 1, 1)
txn_id   = 1

for i, (cid, aid) in enumerate(zip(cust_ids, acct_ids)):
    is_mule = i in mule_idx
    if is_mule:
        for m in range(12):
            for _ in range(rng.integers(3, 9)):
                day      = rng.integers(1, 28)
                txn_date = date(2023, m + 1, day)
                amount   = round(rng.uniform(7800, 9800), 2)
                cpty     = rng.choice(cpty_ids[:30])
                txn_rows.append({'txn_id': f'TXN{txn_id:06d}', 'account_id': aid,
                                 'txn_date': txn_date.isoformat(), 'txn_type': 'CASH_IN',
                                 'amount': amount, 'counterparty_id': cpty})
                txn_id += 1
    else:
        for _ in range(rng.integers(12, 80)):
            txn_date = start + timedelta(days=int(rng.integers(0, 365)))
            txn_type = rng.choice(['CASH_IN','TRANSFER_OUT','TRANSFER_IN','CARD'],
                                   p=[0.15, 0.35, 0.35, 0.15])
            amount   = round(min(rng.lognormal(6.5, 1.2), 50000), 2)
            cpty_pool = cpty_ids[30:] if rng.random() > 0.03 else cpty_ids[:30]
            txn_rows.append({'txn_id': f'TXN{txn_id:06d}', 'account_id': aid,
                             'txn_date': txn_date.isoformat(), 'txn_type': txn_type,
                             'amount': amount, 'counterparty_id': rng.choice(cpty_pool)})
            txn_id += 1

df_txn = (pd.DataFrame(txn_rows)
            .assign(txn_date=lambda d: pd.to_datetime(d['txn_date']))
            .sort_values('txn_date')
            .reset_index(drop=True))
df_txn.to_csv('nb_transactions.csv', index=False)

print(f"✅ Dataset generated")
print(f"   nb_counterparties : {len(df_cpty):>6,} rows")
print(f"   nb_customers      : {len(df_cust):>6,} rows")
print(f"   nb_transactions   : {len(df_txn):>6,} rows")
print(f"   Date range        : {df_txn['txn_date'].min().date()} → {df_txn['txn_date'].max().date()}")

---
## Section 1 — Colab Preview: K-Means Customer Segmentation

> *This section mirrors Section 6.9 of the textbook exactly. Run the cells and compare the output to the printed figures.*

### Why segmentation matters in TM

A single set of rule thresholds applied uniformly to all 500 Northgate customers ignores the fact that customers have radically different normal behaviours. A high-income business owner and a student have different expected cash-in patterns. Segmentation groups customers with similar behavioural profiles, allowing thresholds to be calibrated per segment rather than per the entire population.

### The three behavioural features

For this exercise we use three features, each computed from the transaction history:

| Feature | Description |
|---------|-------------|
| `avg_monthly_cash_in` | Average USD cash deposited per month |
| `txn_frequency` | Total number of transactions in 2023 |
| `cash_ratio` | Fraction of all transactions that are cash deposits |

All three features are standardised (zero mean, unit variance) before clustering. This prevents `avg_monthly_cash_in` — which is in USD — from dominating the distance calculation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df_txn  = pd.read_csv('nb_transactions.csv', parse_dates=['txn_date'])
df_cust = pd.read_csv('nb_customers.csv')

# Build customer-level behavioural features
feats = df_txn.groupby('account_id').agg(
    avg_monthly_cash_in=('amount',
        lambda x: x[df_txn.loc[x.index, 'txn_type'] == 'CASH_IN'].sum() / 12),
    txn_frequency=('txn_id', 'count'),
    cash_ratio=('txn_type', lambda x: (x == 'CASH_IN').mean()),
).fillna(0)

print("Feature matrix — first five accounts:")
print(feats.head().to_string())
print(f"\nShape: {feats.shape[0]} accounts × {feats.shape[1]} features")

In [ ]:
# Standardise before clustering
scaler = StandardScaler()
X = scaler.fit_transform(feats)

# Fit K-Means with K=3
km = KMeans(n_clusters=3, random_state=42, n_init='auto')
feats['segment'] = km.fit_predict(X)

summary = feats.groupby('segment').agg(
    n_customers=('txn_frequency', 'count'),
    avg_cash_in=('avg_monthly_cash_in', 'mean'),
    avg_freq=('txn_frequency', 'mean'),
    avg_cash_ratio=('cash_ratio', 'mean'),
).round(1)

print("K-Means Segment Summary (K=3):")
print(summary.to_string())
print()

MULE_IDS = [f'ACC{i:04d}' for i in range(1, 7)]
mule_segs = feats.loc[feats.index.isin(MULE_IDS), 'segment']
print(f"Mule account segments: {mule_segs.tolist()}")

**What you're seeing:** Three segments have emerged from the Northgate data. One segment contains all six mule accounts. Inspect the segment summary: which segment has the highest average cash-in amount and the highest cash ratio? That is the mule cluster.

Notice that K-Means has no knowledge of which accounts are mules — it discovered the cluster purely from behavioural features. This is the power of unsupervised learning in AML: the algorithm groups customers by behaviour, and compliance officers then label the groups.

In [ ]:
# Scatter plot: avg_monthly_cash_in vs txn_frequency, coloured by segment
colours = ['#4472C4', '#2ECC71', '#F39C12']
fig, ax = plt.subplots(figsize=(7, 5))

for seg in range(3):
    mask = feats['segment'] == seg
    n = mask.sum()
    ax.scatter(feats.loc[mask, 'avg_monthly_cash_in'],
               feats.loc[mask, 'txn_frequency'],
               c=colours[seg], label=f'Segment {seg} (n={n})', alpha=0.6, s=25)

# Highlight mule accounts
mule_feats = feats.loc[feats.index.isin(MULE_IDS)]
ax.scatter(mule_feats['avg_monthly_cash_in'], mule_feats['txn_frequency'],
           c='#E74C3C', marker='*', s=200, zorder=5, label='Mule accounts (★)')

ax.set_xlabel('Avg Monthly Cash In (USD)', fontsize=10)
ax.set_ylabel('Transaction Frequency (annual)', fontsize=10)
ax.set_title('Customer Segmentation — K-Means K=3\n(★ = Northgate mule accounts)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9, framealpha=0.8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

**Reading the chart:** The mule accounts (red stars) form a tight cluster at high average monthly cash-in with moderate transaction frequency. This is the visual signature of structuring: regularly depositing large amounts (but staying sub-threshold on each individual deposit) without the full range of retail banking activity that a genuine high-income customer would show.

---
## Section 2 — Exercise 6.1 Extension: Elbow Method and Segment Labelling

> *This section extends Exercise 6.1. The main-text exercise asks you to run K-Means with K=3 and label the segments. This extension asks you to use the elbow method to justify the choice of K, and to write a formal segment label for each cluster.*

In [ ]:
# Elbow method: inertia (within-cluster sum of squares) for K=2 to K=8
inertias = []
K_range = range(2, 9)

for k in K_range:
    km_k = KMeans(n_clusters=k, random_state=42, n_init='auto')
    km_k.fit(X)
    inertias.append(km_k.inertia_)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(list(K_range), inertias, 'o-', color='#4472C4', linewidth=2)
ax.axvline(3, color='#E74C3C', linestyle='--', linewidth=1.5, label='K=3 (selected)')
ax.set_xlabel('Number of Clusters (K)', fontsize=10)
ax.set_ylabel('Inertia (Within-Cluster SS)', fontsize=10)
ax.set_title('Elbow Method — Northgate Customer Segmentation', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print("Inertia values:")
for k, i in zip(K_range, inertias):
    print(f"  K={k}: {i:,.0f}")

**✏️ YOUR OBSERVATION**

Look at the elbow chart. Is there a clear "elbow" at K=3, or does the inertia continue to drop sharply? 

- If the elbow is at K=3, this supports our choice. Write one sentence explaining why.
- If the elbow is less clear, propose an alternative K and describe what additional segment it might represent.

*Write your answer in Section 3, Question 2.*

In [ ]:
# Segment characterisation: add CRR score and stated income from customer file
feats_enriched = feats.merge(df_cust[['account_id','crr_score','stated_income_usd']],
                              left_index=True, right_on='account_id', how='left')

seg_profile = feats_enriched.groupby('segment').agg(
    n=('txn_frequency', 'count'),
    avg_cash_in=('avg_monthly_cash_in', 'mean'),
    avg_freq=('txn_frequency', 'mean'),
    avg_cash_ratio=('cash_ratio', 'mean'),
    avg_crr=('crr_score', 'mean'),
    avg_income=('stated_income_usd', 'mean'),
).round(1)

print("Enriched Segment Profile (with CRR and stated income):")
print(seg_profile.to_string())

**✏️ YOUR OBSERVATION**

Look at the enriched segment profile. For each of the three segments, propose a business label (e.g. "Low-activity retail", "Moderate transactor", "High-cash structuring risk") and write a one-sentence description of the typical customer in that segment.

*Write your labels in Section 3, Question 3.*

---
## Section 3 — Reflection: Exercise 6.1 Answer Cells

> *Use the cells below to write your answers. Double-click any cell to edit it.*

#### Question 1 — Segment Discovery

*(Edit this cell to write your answer)*

**Which segment number contains all six mule accounts?**  
  
**What are the key feature values (avg cash-in, transaction frequency, cash ratio) that distinguish the mule cluster from the other two segments?**  
  
**What does it mean that K-Means discovered the mule cluster without being told which accounts were mules?**  


#### Question 2 — Choosing K

*(Edit this cell to write your answer)*

**Does the elbow chart support K=3 as the optimal choice? Describe what you see.**  
  
**If you were to run this analysis in a real bank with 500,000 customers, would K=3 still be appropriate? What considerations would change?**  


#### Question 3 — Segment Labelling

*(Edit this cell to write your answer)*

**Segment 0 label and description:**  
  
**Segment 1 label and description:**  
  
**Segment 2 label and description:**  
  
**Which segment's threshold should be tightened for Rule NRB-STRUCT-001, and why?**  


---
## What's Next

In Chapter 7, you will add a second detection rule — **NRB-VEL-002** (rapid-fire transaction velocity) — and tune both rules together. You will also see how the mule cluster from Chapter 6 relates to which accounts trigger multiple rules simultaneously.

Open Chapter 7: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_07.ipynb)